# CNN evaluation HDF5 demo

This notebook evaluates CNN prediction files generated by:

```bash
python cbc_pe/scripts/train_cnn_hdf5.py --config <config.json>
```

The current goal is to compare candidate CNN architectures trained on the 100k HDF5 dataset using the 80/20 train/validation split.

This notebook focuses on:

- global validation metrics
- per-label validation metrics
- standardized-space metrics
- physical-space metrics
- absolute-error quantiles
- detailed diagnostics for a selected model

In [1]:
from pathlib import Path
import os
import sys
import json

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()

# Allow running either from repository root or from cbc_pe/.
if PROJECT_ROOT.name != "cbc_pe" and (PROJECT_ROOT / "cbc_pe").exists():
    PROJECT_ROOT = PROJECT_ROOT / "cbc_pe"

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Main data root for CIEMAT/local office workflow.
DATA_ROOT = Path("/scratch/vserrano/cbc_pe_data")

DATA_PROCESSED = DATA_ROOT / "processed"
DATA_RESULTS = DATA_ROOT / "results"
DATA_MODELS = DATA_ROOT / "models"

dataset_id = "bbh_processed_4s_seobnrv4opt_snr10-25_n100_000"
MODEL_RESULTS_DIR = DATA_RESULTS / dataset_id

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("DATA_PROCESSED:", DATA_PROCESSED)
print("MODEL_RESULTS_DIR:", MODEL_RESULTS_DIR)
print("MODEL_RESULTS_DIR exists:", MODEL_RESULTS_DIR.exists())

PROJECT_ROOT: /afs/ciemat.es/user/v/vserrano/Desktop/gw/Gravitational-Waves-Lab/cbc_pe/notebooks
DATA_ROOT: /scratch/vserrano/cbc_pe_data
DATA_PROCESSED: /scratch/vserrano/cbc_pe_data/processed
MODEL_RESULTS_DIR: /scratch/vserrano/cbc_pe_data/results/bbh_processed_4s_seobnrv4opt_snr10-25_n100_000
MODEL_RESULTS_DIR exists: True


## Register models to compare

Add one entry per trained model.

When a new model finishes training, add its prediction file to `prediction_files` and re-run the comparison cells.

In [3]:
prediction_files = {
    "M00_baseline_emb64": MODEL_RESULTS_DIR / (
        "bbh_processed_4s_seobnrv4opt_snr10-25_n100_000"
        "_SimpleCNN_simple_emb64_mse_MSELoss_seed123"
        "_train_val_predictions_embeddings.npz"
    ),
    "M01_pool1_emb128": MODEL_RESULTS_DIR / (
        "bbh_processed_4s_seobnrv4opt_snr10-25_n100_000"
        "_SimpleCNN_Pool_emb128_pool1_MSELoss_seed123"
        "_train_val_predictions_embeddings.npz"
    ),
    "M02_pool4_emb128": MODEL_RESULTS_DIR / (
        "bbh_processed_4s_seobnrv4opt_snr10-25_n100_000"
        "_SimpleCNN_Pool_emb128_pool4_MSELoss_seed123"
        "_train_val_predictions_embeddings.npz"
    ),
    "M04_pooldeep_emb128_pool4": MODEL_RESULTS_DIR / (
        "bbh_processed_4s_seobnrv4opt_snr10-25_n100_000"
        "_SimpleCNN_PoolDeep_M04_emb128_pool4_deephead_MSELoss_seed123"
        "_train_val_predictions_embeddings.npz"
    ),
    "M06_WideCNN_emb128_pool4": MODEL_RESULTS_DIR / (
        "bbh_processed_4s_seobnrv4opt_snr10-25_n100_000"
        "_WideCNN_Pool_M06_emb128_pool4_widecnn_MSELoss_seed123"
        "_train_val_predictions_embeddings.npz"
    ),
}



for model_id, path in prediction_files.items():
    print(f"{model_id:30s} exists={path.exists()}  file={path.name}")

M00_baseline_emb64             exists=True  file=bbh_processed_4s_seobnrv4opt_snr10-25_n100_000_SimpleCNN_simple_emb64_mse_MSELoss_seed123_train_val_predictions_embeddings.npz
M01_pool1_emb128               exists=True  file=bbh_processed_4s_seobnrv4opt_snr10-25_n100_000_SimpleCNN_Pool_emb128_pool1_MSELoss_seed123_train_val_predictions_embeddings.npz
M02_pool4_emb128               exists=True  file=bbh_processed_4s_seobnrv4opt_snr10-25_n100_000_SimpleCNN_Pool_emb128_pool4_MSELoss_seed123_train_val_predictions_embeddings.npz
M04_pooldeep_emb128_pool4      exists=True  file=bbh_processed_4s_seobnrv4opt_snr10-25_n100_000_SimpleCNN_PoolDeep_M04_emb128_pool4_deephead_MSELoss_seed123_train_val_predictions_embeddings.npz
M06_WideCNN_emb128_pool4       exists=True  file=bbh_processed_4s_seobnrv4opt_snr10-25_n100_000_WideCNN_Pool_M06_emb128_pool4_widecnn_MSELoss_seed123_train_val_predictions_embeddings.npz


## Metric helper functions

In [4]:
def regression_metrics(y_true, y_pred):
    residual = y_true - y_pred

    mse = np.mean(residual**2, axis=0)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(residual), axis=0)
    bias = np.mean(residual, axis=0)
    residual_std = np.std(residual, axis=0)

    ss_res = np.sum(residual**2, axis=0)
    ss_tot = np.sum((y_true - np.mean(y_true, axis=0))**2, axis=0)
    r2 = 1.0 - ss_res / ss_tot

    global_mse = np.mean(residual**2)
    global_rmse = np.sqrt(global_mse)
    global_mae = np.mean(np.abs(residual))

    return {
        "mse": mse,
        "rmse": rmse,
        "mae": mae,
        "bias": bias,
        "residual_std": residual_std,
        "r2": r2,
        "global_mse": global_mse,
        "global_rmse": global_rmse,
        "global_mae": global_mae,
    }


def abs_error_quantiles(y_true, y_pred, label_names, model_id, space):
    abs_err = np.abs(y_true - y_pred)
    rows = []

    for j, label in enumerate(label_names):
        q50, q90, q95, q99 = np.quantile(abs_err[:, j], [0.50, 0.90, 0.95, 0.99])

        rows.append({
            "model_id": model_id,
            "space": space,
            "label": label,
            "q50_abs_error": q50,
            "q90_abs_error": q90,
            "q95_abs_error": q95,
            "q99_abs_error": q99,
            "max_abs_error": abs_err[:, j].max(),
        })

    return rows


def get_label_names(data):
    if "label_names" in data.files:
        return [str(x) for x in data["label_names"].tolist()]
    return ["chirp_mass", "total_mass", "chi_eff"]

## Compute validation metrics for all models

Metrics are computed in:

- standardized space
- physical space

In [5]:
metric_rows = []
quantile_rows = []

for model_id, path in prediction_files.items():
    if not path.exists():
        print(f"Skipping missing file: {model_id} -> {path}")
        continue

    data = np.load(path, allow_pickle=True)

    pred_val = data["pred_val"]
    y_val = data["y_val"]

    y_mean = data["y_mean"]
    y_std = data["y_std"]
    label_names = get_label_names(data)

    # -------------------------
    # Standardized space
    # -------------------------
    metrics_std = regression_metrics(y_val, pred_val)

    metric_rows.append({
        "model_id": model_id,
        "space": "standardized",
        "label": "global",
        "MSE": metrics_std["global_mse"],
        "RMSE": metrics_std["global_rmse"],
        "MAE": metrics_std["global_mae"],
        "Bias": np.nan,
        "Residual std": np.nan,
        "R2": np.nan,
    })

    for j, label in enumerate(label_names):
        metric_rows.append({
            "model_id": model_id,
            "space": "standardized",
            "label": label,
            "MSE": metrics_std["mse"][j],
            "RMSE": metrics_std["rmse"][j],
            "MAE": metrics_std["mae"][j],
            "Bias": metrics_std["bias"][j],
            "Residual std": metrics_std["residual_std"][j],
            "R2": metrics_std["r2"][j],
        })

    quantile_rows.extend(
        abs_error_quantiles(
            y_true=y_val,
            y_pred=pred_val,
            label_names=label_names,
            model_id=model_id,
            space="standardized",
        )
    )

    # -------------------------
    # Physical space
    # -------------------------
    y_val_phys = y_val * y_std + y_mean
    pred_val_phys = pred_val * y_std + y_mean

    metrics_phys = regression_metrics(y_val_phys, pred_val_phys)

    metric_rows.append({
        "model_id": model_id,
        "space": "physical",
        "label": "global",
        "MSE": metrics_phys["global_mse"],
        "RMSE": metrics_phys["global_rmse"],
        "MAE": metrics_phys["global_mae"],
        "Bias": np.nan,
        "Residual std": np.nan,
        "R2": np.nan,
    })

    for j, label in enumerate(label_names):
        metric_rows.append({
            "model_id": model_id,
            "space": "physical",
            "label": label,
            "MSE": metrics_phys["mse"][j],
            "RMSE": metrics_phys["rmse"][j],
            "MAE": metrics_phys["mae"][j],
            "Bias": metrics_phys["bias"][j],
            "Residual std": metrics_phys["residual_std"][j],
            "R2": metrics_phys["r2"][j],
        })

    quantile_rows.extend(
        abs_error_quantiles(
            y_true=y_val_phys,
            y_pred=pred_val_phys,
            label_names=label_names,
            model_id=model_id,
            space="physical",
        )
    )

summary_df = pd.DataFrame(metric_rows)
quantiles_df = pd.DataFrame(quantile_rows)

summary_df

,model_id,space,label,MSE,RMSE,MAE,Bias,Residual std,R2
0,M00_baseline_emb64,standardized,global,0.177084,0.420813,0.305378,NaN,NaN,NaN
1,M00_baseline_emb64,standardized,chirp_mass,0.140808,0.375244,0.268875,-0.013185,0.375013,0.857300
2,M00_baseline_emb64,standardized,total_mass,0.112468,0.335363,0.249743,-0.009372,0.335232,0.885663
3,M00_baseline_emb64,standardized,chi_eff,0.277973,0.527231,0.397517,-0.007467,0.527178,0.725214
4,M00_baseline_emb64,physical,global,58.075588,7.620734,4.431233,NaN,NaN,NaN
5,M00_baseline_emb64,physical,chirp_mass,38.462017,6.201776,4.443779,-0.217914,6.197942,0.857299
6,M00_baseline_emb64,physical,total_mass,135.710419,11.649482,8.675298,-0.325544,11.644970,0.885663
7,M00_baseline_emb64,physical,chi_eff,0.053663,0.231652,0.174658,-0.003281,0.231628,0.725214
8,M01_pool1_emb128,standardized,global,0.178520,0.422516,0.310376,NaN,NaN,NaN
9,M01_pool1_emb128,standardized,chirp_mass,0.143446,0.378742,0.277267,0.030674,0.377497,0.854627


## Global validation metrics

In [6]:
summary_df.query("space == 'standardized' and label == 'global'").sort_values("MSE")

,model_id,space,label,MSE,RMSE,MAE,Bias,Residual std,R2
24,M04_pooldeep_emb128_pool4,standardized,global,0.176160,0.419714,0.305510,NaN,NaN,NaN
0,M00_baseline_emb64,standardized,global,0.177084,0.420813,0.305378,NaN,NaN,NaN
8,M01_pool1_emb128,standardized,global,0.178520,0.422516,0.310376,NaN,NaN,NaN
32,M06_WideCNN_emb128_pool4,standardized,global,0.180994,0.425434,0.311171,NaN,NaN,NaN
16,M02_pool4_emb128,standardized,global,0.182914,0.427684,0.315345,NaN,NaN,NaN


## Per-label validation metrics: standardized space

In [7]:
summary_df.query("space == 'standardized' and label != 'global'").sort_values(["label", "MSE"])

,model_id,space,label,MSE,RMSE,MAE,Bias,Residual std,R2
3,M00_baseline_emb64,standardized,chi_eff,0.277973,0.527231,0.397517,-0.007467,0.527178,0.725214
11,M01_pool1_emb128,standardized,chi_eff,0.279148,0.528344,0.401955,-0.000254,0.528344,0.724053
27,M04_pooldeep_emb128_pool4,standardized,chi_eff,0.279810,0.528971,0.400771,-0.038120,0.527596,0.723398
35,M06_WideCNN_emb128_pool4,standardized,chi_eff,0.283660,0.532597,0.405834,-0.004616,0.532578,0.719592
19,M02_pool4_emb128,standardized,chi_eff,0.283983,0.532901,0.406288,-0.014028,0.532716,0.719272
25,M04_pooldeep_emb128_pool4,standardized,chirp_mass,0.137081,0.370245,0.264898,-0.019893,0.369710,0.861077
1,M00_baseline_emb64,standardized,chirp_mass,0.140808,0.375244,0.268875,-0.013185,0.375013,0.857300
9,M01_pool1_emb128,standardized,chirp_mass,0.143446,0.378742,0.277267,0.030674,0.377497,0.854627
33,M06_WideCNN_emb128_pool4,standardized,chirp_mass,0.144056,0.379548,0.273944,0.019206,0.379062,0.854008
17,M02_pool4_emb128,standardized,chirp_mass,0.147499,0.384056,0.280664,0.000235,0.384056,0.850520


## Per-label validation metrics: physical space

In [8]:
summary_df.query("space == 'physical' and label != 'global'").sort_values(["label", "RMSE"])

,model_id,space,label,MSE,RMSE,MAE,Bias,Residual std,R2
7,M00_baseline_emb64,physical,chi_eff,0.053663,0.231652,0.174658,-0.003281,0.231628,0.725214
15,M01_pool1_emb128,physical,chi_eff,0.053890,0.232141,0.176609,-0.000112,0.232141,0.724053
31,M04_pooldeep_emb128_pool4,physical,chi_eff,0.054017,0.232416,0.176089,-0.016749,0.231812,0.723399
39,M06_WideCNN_emb128_pool4,physical,chi_eff,0.054760,0.234009,0.178313,-0.002028,0.234001,0.719593
23,M02_pool4_emb128,physical,chi_eff,0.054823,0.234143,0.178512,-0.006164,0.234062,0.719274
29,M04_pooldeep_emb128_pool4,physical,chirp_mass,37.443871,6.119140,4.378024,-0.328783,6.110294,0.861076
5,M00_baseline_emb64,physical,chirp_mass,38.462017,6.201776,4.443779,-0.217914,6.197942,0.857299
13,M01_pool1_emb128,physical,chirp_mass,39.182098,6.259561,4.582464,0.506956,6.239001,0.854627
37,M06_WideCNN_emb128_pool4,physical,chirp_mass,39.349358,6.272907,4.527555,0.317423,6.264858,0.854007
21,M02_pool4_emb128,physical,chirp_mass,40.289371,6.347391,4.638609,0.003882,6.347382,0.850519


## Absolute-error quantiles

These are useful for checking whether a model improves only the mean error or also the tails.

In [9]:
quantiles_df.query("space == 'physical'").sort_values(["label", "q90_abs_error"])

,model_id,space,label,q50_abs_error,q90_abs_error,q95_abs_error,q99_abs_error,max_abs_error
5,M00_baseline_emb64,physical,chi_eff,0.135971,0.372139,0.470430,0.699990,1.240961
23,M04_pooldeep_emb128_pool4,physical,chi_eff,0.138944,0.373626,0.470482,0.702085,1.370715
17,M02_pool4_emb128,physical,chi_eff,0.142903,0.376776,0.469615,0.702590,1.301121
11,M01_pool1_emb128,physical,chi_eff,0.138833,0.376889,0.470462,0.706640,1.237622
29,M06_WideCNN_emb128_pool4,physical,chi_eff,0.140824,0.379565,0.473969,0.695894,1.190920
21,M04_pooldeep_emb128_pool4,physical,chirp_mass,3.122549,9.802851,12.835625,20.175450,42.034042
3,M00_baseline_emb64,physical,chirp_mass,3.171960,9.824257,13.072489,20.620085,42.305740
27,M06_WideCNN_emb128_pool4,physical,chirp_mass,3.235892,10.054593,13.216316,20.460182,39.005211
9,M01_pool1_emb128,physical,chirp_mass,3.373236,10.111572,13.052073,19.967679,45.330147
15,M02_pool4_emb128,physical,chirp_mass,3.401598,10.290470,13.379850,20.279280,37.187252


## Save comparison tables

In [ ]:
comparison_dir = MODEL_RESULTS_DIR / "evaluation"
comparison_dir.mkdir(parents=True, exist_ok=True)

summary_csv = comparison_dir / "architecture_search_val_metrics.csv"
quantiles_csv = comparison_dir / "architecture_search_val_abs_error_quantiles.csv"

summary_df.to_csv(summary_csv, index=False)
quantiles_df.to_csv(quantiles_csv, index=False)

print("Saved:", summary_csv)
print("Saved:", quantiles_csv)

## Detailed diagnostics for one selected model

Use this section to inspect residuals, prediction-vs-truth plots, SNR dependence, and embedding behavior for one model.

Do not run detailed plots for every model unless the summary tables justify it.

In [ ]:
SELECTED_MODEL_ID = "M00_baseline_emb64"
# SELECTED_MODEL_ID = "M01_pool1_emb128"
# SELECTED_MODEL_ID = "M02_pool4_emb128"
# SELECTED_MODEL_ID = "M04_pooldeep_emb128_pool4"

selected_path = prediction_files[SELECTED_MODEL_ID]

data = np.load(selected_path, allow_pickle=True)

pred_train = data["pred_train"]
y_train = data["y_train"]
emb_train = data["emb_train"]

pred_val = data["pred_val"]
y_val = data["y_val"]
emb_val = data["emb_val"]

y_mean = data["y_mean"]
y_std = data["y_std"]
label_names = get_label_names(data)

# Robust index loading across script versions.
train_idx = data["idx_train"] if "idx_train" in data.files else data["train_idx"]
val_idx = data["idx_val"] if "idx_val" in data.files else data["val_idx"]

pred_val_phys = pred_val * y_std + y_mean
y_val_phys = y_val * y_std + y_mean
residual_val_phys = y_val_phys - pred_val_phys

print("Selected model:", SELECTED_MODEL_ID)
print("prediction file:", selected_path)
print("label_names:", label_names)
print("pred_val:", pred_val.shape)
print("y_val:", y_val.shape)
print("emb_val:", emb_val.shape)
print("val_idx:", val_idx.shape)

In [ ]:
for j, label in enumerate(label_names):
    true = y_val_phys[:, j]
    pred = pred_val_phys[:, j]

    plt.figure(figsize=(5.5, 5))
    hb = plt.hexbin(true, pred, gridsize=70, mincnt=1, bins="log")
    plt.colorbar(hb, label="log10(count)")

    lo = min(true.min(), pred.min())
    hi = max(true.max(), pred.max())
    plt.plot([lo, hi], [lo, hi], linestyle="--")

    plt.xlabel(f"True {label}")
    plt.ylabel(f"Predicted {label}")
    plt.title(f"{SELECTED_MODEL_ID}: predicted vs true ({label})")
    plt.grid(True)
    plt.show()

In [ ]:
for j, label in enumerate(label_names):
    true = y_val_phys[:, j]
    residual = residual_val_phys[:, j]

    plt.figure(figsize=(6, 4.5))
    hb = plt.hexbin(true, residual, gridsize=70, mincnt=1, bins="log")
    plt.colorbar(hb, label="log10(count)")

    plt.axhline(0.0, linestyle="--")
    plt.xlabel(f"True {label}")
    plt.ylabel("Residual: true - predicted")
    plt.title(f"{SELECTED_MODEL_ID}: residual vs true ({label})")
    plt.grid(True)
    plt.show()

## Comparison between models

In [ ]:
for label_idx, label in enumerate(label_names):
    plt.figure(figsize=(7, 4))

    for model_id, path in prediction_files.items():
        if not path.exists():
            continue

        data = np.load(path, allow_pickle=True)
        pred_val = data["pred_val"]
        y_val = data["y_val"]
        y_mean = data["y_mean"]
        y_std = data["y_std"]

        pred_phys = pred_val * y_std + y_mean
        y_phys = y_val * y_std + y_mean
        residual = y_phys[:, label_idx] - pred_phys[:, label_idx]

        plt.hist(
            residual,
            bins=100,
            density=True,
            histtype="step",
            linewidth=1.5,
            label=model_id,
        )

    plt.axvline(0.0, linestyle="--")
    plt.xlabel(f"Residual: true - predicted ({label})")
    plt.ylabel("Density")
    plt.title(f"Validation residual comparison ({label})")
    plt.legend()
    plt.grid(True)
    plt.show()